# Breast Cancer Diagnosis Prediction using Machine Learning

**Project objective:** Build a supervised machine learning model that predicts whether a breast tumor is **benign** or **malignant** using diagnostic cell-nucleus measurements.

**Dataset:** Breast Cancer Wisconsin Diagnostic dataset from the UCI Machine Learning Repository, available through `sklearn.datasets.load_breast_cancer`.

In [ ]:
# 1. Import libraries
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, RocCurveDisplay
from sklearn.inspection import permutation_importance

In [ ]:
# 2. Load dataset from a secondary source
cancer = load_breast_cancer(as_frame=True)
df = cancer.frame.copy()
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})

print('Rows:', df.shape[0])
print('Features:', len(cancer.feature_names))
df.head()

In [ ]:
# 3. Basic data understanding
print(df['diagnosis'].value_counts())
print('
Missing values:', df.isnull().sum().sum())
df.describe().T.head(10)

In [ ]:
# 4. Visualize class distribution
counts = df['diagnosis'].value_counts().reindex(['Benign', 'Malignant'])
plt.figure(figsize=(6,4))
plt.bar(counts.index, counts.values)
plt.title('Diagnosis Distribution')
plt.ylabel('Number of Samples')
plt.show()

In [ ]:
# 5. Visualize key feature differences
features = ['mean radius','mean texture','mean perimeter','mean area','mean concave points']
means = df.groupby('diagnosis')[features].mean().T

plt.figure(figsize=(8,5))
x = np.arange(len(features))
w = 0.38
plt.bar(x - w/2, means['Benign'], w, label='Benign')
plt.bar(x + w/2, means['Malignant'], w, label='Malignant')
plt.xticks(x, [f.replace('mean ', '') for f in features], rotation=25, ha='right')
plt.title('Average Cell Nucleus Features by Diagnosis')
plt.ylabel('Mean Value')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 6. Prepare train/test split
X = df[cancer.feature_names]
y = df['target']  # 0 = malignant, 1 = benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training samples:', X_train.shape[0])
print('Testing samples:', X_test.shape[0])

In [ ]:
# 7. Define models: single models + ensemble model
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=5000, random_state=42))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=250, random_state=42, class_weight='balanced'
    ),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
}

models['Soft Voting Ensemble'] = VotingClassifier(
    estimators=[
        ('lr', models['Logistic Regression']),
        ('rf', models['Random Forest']),
        ('gb', models['Gradient Boosting'])
    ],
    voting='soft'
)

In [ ]:
# 8. Train and evaluate models
results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, predictions),
        'Precision': precision_score(y_test, predictions),
        'Recall': recall_score(y_test, predictions),
        'F1-score': f1_score(y_test, predictions),
        'ROC-AUC': roc_auc_score(y_test, probabilities),
        'CV Accuracy Mean': cross_val_score(model, X, y, cv=5, scoring='accuracy').mean()
    })

metrics = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
metrics

In [ ]:
# 9. Visualize model performance
plt.figure(figsize=(10,5))
x = np.arange(len(metrics))
width = 0.18
for i, col in enumerate(['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC']):
    plt.bar(x + (i-2)*width, metrics[col], width, label=col)
plt.xticks(x, metrics['Model'], rotation=25, ha='right')
plt.ylim(0.85, 1.01)
plt.ylabel('Score')
plt.title('Model Performance Comparison')
plt.legend(ncols=3)
plt.tight_layout()
plt.show()

In [ ]:
# 10. Confusion matrix for the best model
best_model_name = metrics.iloc[0]['Model']
best_model = fitted_models[best_model_name]

pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, pred)

plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title(f'Confusion Matrix: {best_model_name}')
plt.xticks([0,1], ['Malignant','Benign'])
plt.yticks([0,1], ['Malignant','Benign'])
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha='center', va='center', fontsize=16)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.colorbar()
plt.show()

In [ ]:
# 11. ROC curve comparison
plt.figure(figsize=(7,5))
for name, model in fitted_models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=plt.gca())
plt.title('ROC Curves')
plt.show()

In [ ]:
# 12. Feature importance using permutation importance
importance = permutation_importance(
    best_model, X_test, y_test, n_repeats=15, random_state=42, scoring='roc_auc'
)

imp_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importance.importances_mean
}).sort_values('Importance', ascending=False).head(10)

plt.figure(figsize=(8,5))
plt.barh(imp_df['Feature'][::-1], imp_df['Importance'][::-1])
plt.title(f'Top Predictive Features ({best_model_name})')
plt.xlabel('Permutation Importance')
plt.tight_layout()
plt.show()

imp_df

## Final Interpretation

The model predicts breast cancer diagnosis from numerical measurements extracted from cell nuclei images. The best-performing model achieved strong classification performance on the test set, meaning it can separate benign and malignant cases well on this dataset.

**Important note:** This project is for academic demonstration only. A real clinical deployment would require external validation, medical review, privacy controls, and approval by healthcare professionals.

In [ ]:
# 13. Save the best model for optional deployment
import joblib
joblib.dump(best_model, 'breast_cancer_best_model.joblib')
print('Saved:', best_model_name)